In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import community.community_louvain as community_louvain

excel_file_path = 'Menstrual_cramps.xlsx'

node_df = pd.read_excel(excel_file_path, sheet_name='경혈단독_count')
node_df['경혈명'] = node_df['경혈명'].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)

invalid_nodes = ['non-acupoint', 'ashi-point', 'BV', 'CV', 'FV']

valid_nodes = node_df[~node_df['경혈명'].isin(invalid_nodes)]['경혈명'].tolist()

edge_df = pd.read_excel(excel_file_path, sheet_name='경혈pair_count')

cols_to_clean = ['Source', 'Target']
for col in cols_to_clean:
    edge_df[col] = edge_df[col].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)

edge_df = edge_df[edge_df['Source'].isin(valid_nodes) & edge_df['Target'].isin(valid_nodes)]

G = nx.Graph()
G.add_nodes_from(valid_nodes) # 유효 노드만 추가
for _, row in edge_df.iterrows():
    G.add_edge(row['Source'], row['Target'], weight=float(row['Weight']))


# 1. Louvain 군집화 실행 (전체 그래프 G 대상)

partition = community_louvain.best_partition(G, weight='weight', resolution=1.0, random_state=42)
num_communities = len(set(partition.values()))

print(f"전체 네트워크 군집 분석 완료: 총 {num_communities}개 그룹 탐지됨")

# 2. 군집 중심을 원형으로 강제 배치
community_pos = {}
radius = 4.0  # 3.0 → 4.0 (군집 간 거리 증가)

for i in range(num_communities):
    angle = 2 * np.pi * i / num_communities
    community_pos[i] = (radius * np.cos(angle), radius * np.sin(angle))

# 3. 각 군집 내부 노드 배치 (겹침 최소화)
pos = {}
for comm_id in range(num_communities):
    comm_nodes = [n for n in G.nodes() if partition[n] == comm_id]
    comm_subgraph = G.subgraph(comm_nodes)

    if len(comm_nodes) == 1:
        pos[comm_nodes[0]] = community_pos[comm_id]
    else:
        # k값 증가 (노드 간 반발력 강화), iterations 증가
        subgraph_pos = nx.spring_layout(
            comm_subgraph,
            k=3,
            seed=42,
            iterations=100
        )

        center = community_pos[comm_id]
        scale = 0.9

        for node in comm_nodes:
            pos[node] = (
                center[0] + subgraph_pos[node][0] * scale,
                center[1] + subgraph_pos[node][1] * scale
            )

# 4. 노드 크기 설정 및 리더(Leader) 선정
degree_dict = dict(G.degree())
node_sizes = []
leader_info = {}
community_groups = {}

for node, group_id in partition.items():
    if group_id not in community_groups:
        community_groups[group_id] = []
    community_groups[group_id].append(node)

for group_id, members in community_groups.items():
    leader = max(members, key=lambda x: degree_dict[x])
    leader_info[group_id] = leader

for node in G.nodes():
    group_id = partition[node]
    if node == leader_info[group_id]:
        node_sizes.append(1300)
    else:
        node_sizes.append(500)

# 5. 시각화 그리기
plt.figure(figsize=(22, 22))

cmap = cm.get_cmap('tab10', num_communities)
if num_communities > 10:
    cmap = cm.get_cmap('tab20', num_communities)

internal_edges = [(u, v) for u, v in G.edges() if partition[u] == partition[v]]
nx.draw_networkx_edges(G, pos, edgelist=internal_edges,
                       width=1.5, alpha=0.3, edge_color='#888888')

external_edges = [(u, v) for u, v in G.edges() if partition[u] != partition[v]]
nx.draw_networkx_edges(G, pos, edgelist=external_edges,
                       width=0.5, alpha=0.3, edge_color='#888888')

node_colors = [cmap(partition[node]) for node in G.nodes()]
nx.draw_networkx_nodes(G, pos,
                       node_color=node_colors,
                       node_size=node_sizes,
                       alpha=0.9,
                       edgecolors='white',
                       linewidths=1.5)

nx.draw_networkx_labels(G, pos, font_size=9, font_family='sans-serif', font_weight='bold')

# 6. Legend (범례)
legend_handles = []

for i in sorted(leader_info.keys()):
    group_color = cmap(i)
    leader_name = leader_info[i]
    count = len(community_groups[i])
    label_text = f"Cluster {i+1} (Leader: {leader_name}, n={count})"
    patch = mpatches.Patch(color=group_color, label=label_text)
    legend_handles.append(patch)

plt.legend(handles=legend_handles,
           title="Cluster Info (Leader & Size)",
           loc='upper left',
           bbox_to_anchor=(1, 1),
           fontsize=11,
           title_fontsize=13,
           frameon=True,
           shadow=True)

plt.title("Louvain Clustering", fontsize=22, fontweight='bold', pad=20)
plt.axis('off')
plt.tight_layout()

plt.savefig("Louvain_Clustering_final.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Louvain 군집별 소속 경혈 리스트 출력
print(f"=== Louvain 군집 분석 상세 결과 (총 {num_communities}개 군집) ===\n")

# 그룹 ID 순서대로 정렬하여 출력
for i in sorted(community_groups.keys()):
    leader_name = leader_info[i]
    members = community_groups[i]
    count = len(members)

    sorted_members = sorted(members)

    print(f" Cluster {i+1}")
    print(f"  - 대표 경혈(Leader): {leader_name}")
    print(f"  - 소속 경혈 수: {count}개")
    print(f"  - 경혈 리스트: {', '.join(sorted_members)}")
    print("-" * 50)

import pandas as pd

cluster_data = []
for i in sorted(community_groups.keys()):
    cluster_data.append({
        'Cluster': f"Cluster {i+1}",
        'Leader': leader_info[i],
        'Count': len(community_groups[i]),
        'Members': ', '.join(sorted(community_groups[i]))
    })

df_cluster_summary = pd.DataFrame(cluster_data)